We all sometimes wish the world looked a bit different, fortunately statistics has the answer! I'm going to detail some useful statistical tools to draw conclusions about data from a different distribution than your empirical data generating distribution. This allows us to answer *what if* questions and reason about counterfactual outcomes.

### Importance sampling

Imagine we want to find an expectation involving some r.v. $X \sim P$ but we don't know how to sample from $P$ (or similarly, we don't have data). If we know the pdf $p(x)$ there is a neat trick to calculate it using samples from another distribution $Q$.

$$\mathbb{E}_P[f(X)] = \int f(x) dP = \int f(x) p(x) dx = \int f(x) \frac{p(x)}{q(x)} q(x) dx = \int f(x) \frac{p(x)}{q(x)} dQ = \mathbb{E}_Q[f(Z)\frac{p(Z)}{q(Z)}].$$

In other words, to calculate $\mathbb{E}[f(X)]$ where $X \sim P$, draw $N$ samples $z_i$ from another distribution $Q$ and calculate $\mathbb{E}[f(Z)\frac{p(Z)}{q(Z)}]\approx \frac{1}{N}\sum_i f(z_i) \frac{p(z_i)}{q(z_i)}$.

A variant of this, self-normalized importance sampling, also works if you only know $p(x) = c p_0(x)$ up to a normalizing constant. Common with Bayesian posteriors.

> The density ratio $\frac{p}{q}$ is also known as the Radon-Nikodym derivative of the measure $P$ w.r.t. $Q$, we can see this family of techniques as a change of measure. If you're not familiar with the notation $dP$, we can formally let $dP=\frac{dP}{dx}dx=p(x)dx$. The pdf $p$ is the Radon-Nikodym derivative of the measure/cdf $P$ w.r.t. the Lebesgue measure on the real line, the measure of a uniform distribution. On $\mathbb{R}$ the Radon-Nikodym derivative $\frac{p}{q}$ exists if the support of $Q$ covers the support of $P$.

If $X \sim \mathcal{N}(0,1)$ find $P(X>10)$. Sampling from the standard normal is of course easy, but we would quickly notice that we would record vanishingly few samples above 10. So instead we sample from a shifted exponential $Z \sim \text{Exp}(1)$ with importance sampling.

$$P(X>10) = \mathbb{E}_\mathcal{N}[\mathbb{1}_{X>10}] = \mathbb{E}_{\text{Exp}}[\frac{e^{-\frac{(Z+10)^2}{2}}}{\sqrt{2 \pi} e^{-Z}}] = \mathbb{E}_{\text{Exp}}[\frac{e^{Z-\frac{(Z+10)^2}{2}}}{\sqrt{2 \pi}}].$$

In [1]:
import numpy as np
N= 100_000
z = np.random.exponential(1,N)
f = np.exp(z - 0.5 * (z+10)**2 - 0.5 * np.log(2 * np.pi))
print(f"mean: {np.mean(f):.2e}, std: {np.std(f):.2e}")

mean: 7.62e-24, std: 1.59e-23


Variance is quite high for this estimator, but if the alternative is to draw >> $10^{24}$ samples...

> Fun fact, people say the silver crash of 2026 was a $10\sigma$-event. Very unfortunate.

### Inverse Probability Weighting




A particularly useful application is when your target distribution is uniform. If you have confounded data on interventions $T$, outcomes $Y$, and features/covariates $X$ you may ask the question *what would this data have looked like if I did an A/B test*?

In this case you would imagine a world where treatments are randomized everywhere. Your data generating distribution can be factored as 
$$q(Y,T,X)=q(Y|T,X)q(T|X)q(X)$$
In an A/B setting your target treatment policy $q(T|X)$ would be a uniform distribution. In our importance sampling formula we can simply set the numerator to a constant.

IPW can be applied to general machine learning algorithms by weighting each datapoint by $\frac{1}{q(T=\text{observed}|X)}$, using some classifier as treatment propensity model. This model would now approximate learning from an imagined unconfounded dataset free from treatment bias.

### Control Variates

Not directly related to the premise but another useful technique, to decrease the variance of an estimator.

If we want to estimate $\mathbb{E}[f(X)]$ and have another correlated r.v. $h(X)$ available with known mean $\mu_h = \mathbb{E}[h(X)]$, we can construct a consistent unbiased estimator as $$\mathbb{E}[\tilde{f}(X)] =  \mathbb{E}[f(X) + \theta (h(X) - \mu_h)].$$ This estimator has variance  $$\mathbb{V}[\tilde{f}(X)]=\mathbb{V}[{f}(X)] + \theta^2\mathbb{V}[{h}(X)] + 2\theta \text{Cov}({f}(X), {h}(X)).$$

With minimal variance at $\theta = -\frac{\text{Cov}({f}(X), {h}(X))}{\mathbb{V}[{h}(X)]}.$

The maximal variance reduction is $\frac{\mathbb{V}[{\tilde{f}}(X)]}{\mathbb{V}[{f}(X)]} = 1- \rho^2(f,h).$

Let's get back to $P(X>10)$ and try to decrease the variance. I think a good control variate would be $h = \log(Z)$, with mean $\mathbb{E}[\log(Z)] = -\gamma$ [The Euler-Mascheroni constant](https://en.wikipedia.org/wiki/Euler%27s_constant#Integrals). Let me know if you can find a better control variate!

In [5]:
from statsmodels.regression.linear_model import OLS
h = np.log(z)
# Estimate optimal theta
theta = -OLS(f,h).fit().params[0]
mu_h = -np.euler_gamma
f_cv = f + theta * (h-mu_h)
print(f"mean: {np.mean(f_cv):.2e}, std: {np.std(f_cv):.2e}")

mean: 7.63e-24, std: 7.94e-24


A common variant of this is CUPED where $h$ is the target $Y$ measured for each unit before the data period. In this case the mean $\mathbb{E}[h]$ is unknown, but it will cancel out for estimates of differences like ATE, CATE.

### Doubly Robust Estimators

Like CUPED these are relevant to causal estimands, for example the Conditional Average Treatment Effect $\mathbb{E}[Y^1 - Y^0 | X]$. We could of course build separate models for the counterfactual treatment cases, or include the treatment as a feature, but do you trust a general purpose machine learning algorithm to put special attention to the one thing you're interested in, the treatment effect?

We can introduce a control variate in the IPW estimator as $\mathbb E[\frac{1}{P(T=t_i | X)}f(X)] = \mathbb E[\frac{1}{P(T=t_i | X)}f(X) - (\frac{\mathbf{1}_{T=t_i}}{P(T=t_i | X)}-1)\mu(X)]$ where $\mu(X)$ is some or any informed guess, an outcome model. In practice this is a 3 stage process: estimate a propensity model $e(x) = P(T=t_i | X)$, an outcome model $\mu(X) = f(X)$, construct an unbiased estimator of the outcome $Y$ as.

### Off-policy Evaluation